In [74]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

data = pd.read_csv("../data/processed/analysis_dataset.csv", index_col=0, parse_dates=True)
data.head()

,BAC,GS,JPM,SP500,DGS10,DGS2,VIX,ret_JPM,ret_BAC,ret_GS,ret_SP500,d_DGS10,d_DGS2,d_VIX
2015-01-05,13.573362,149.305878,44.636642,2020.579956,2.04,0.68,19.920000,-0.031537,-0.029481,-0.031720,-0.018447,-0.08,0.02,2.129999
2015-01-06,13.167254,146.285431,43.479256,2002.609985,1.97,0.65,21.120001,-0.026271,-0.030376,-0.020437,-0.008933,-0.07,-0.03,1.200001
2015-01-07,13.229733,148.465546,43.545605,2025.900024,1.96,0.62,19.309999,0.001525,0.004734,0.014793,0.011563,-0.01,-0.03,-1.810001
2015-01-08,13.503080,150.835876,44.518688,2062.139893,2.03,0.62,17.010000,0.022100,0.020451,0.015839,0.017730,0.07,0.00,-2.299999
2015-01-09,13.260975,148.521027,43.744637,2044.810059,1.98,0.59,17.549999,-0.017540,-0.018092,-0.015466,-0.008439,-0.05,-0.03,0.539999


In [75]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    return {
        "variable": name,
        "adf_stat": result[0],
        "p_value": result[1],
        "stationary": result[1] < 0.05
    }

variables_to_test = ["ret_JPM", "ret_BAC", "ret_GS", "ret_SP500", "d_DGS10", "d_DGS2", "d_VIX"]
adf_results = pd.DataFrame([adf_test(data[v], v) for v in variables_to_test])
adf_results

,variable,adf_stat,p_value,stationary
0,ret_JPM,-15.610212,1.783739e-28,True
1,ret_BAC,-17.248309,6.045938e-30,True
2,ret_GS,-17.103653,7.447393e-30,True
3,ret_SP500,-17.471795,4.538542e-30,True
4,d_DGS10,-40.533721,0.000000e+00,True
5,d_DGS2,-8.270828,4.798534e-13,True
6,d_VIX,-12.521618,2.541026e-23,True


In [76]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    return {
        "variable": name,
        "adf_stat": result[0],
        "p_value": result[1],
        "stationary": result[1] < 0.05
    }

variables_to_test = ["ret_JPM", "ret_BAC", "ret_GS", "ret_SP500", "d_DGS10", "d_DGS2", "d_VIX"]
adf_results = pd.DataFrame([adf_test(data[v], v) for v in variables_to_test])
adf_results

,variable,adf_stat,p_value,stationary
0,ret_JPM,-15.610212,1.783739e-28,True
1,ret_BAC,-17.248309,6.045938e-30,True
2,ret_GS,-17.103653,7.447393e-30,True
3,ret_SP500,-17.471795,4.538542e-30,True
4,d_DGS10,-40.533721,0.000000e+00,True
5,d_DGS2,-8.270828,4.798534e-13,True
6,d_VIX,-12.521618,2.541026e-23,True


In [77]:
from linearmodels.system import SUR

equations = {
    "JPM": {"dependent": data["ret_JPM"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "BAC": {"dependent": data["ret_BAC"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "GS": {"dependent": data["ret_GS"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
}

sur_model = SUR(equations)
sur_results = sur_model.fit(cov_type="kernel", kernel="bartlett", bandwidth=5)
print(sur_results)

                           System GLS Estimation Summary                           
Estimator:                        GLS   Overall R-squared:                   0.5454
No. Equations.:                     3   McElroy's R-squared:                 0.3525
No. Observations:                2914   Judge's (OLS) R-squared:             0.5454
Date:                Sat, Sep 05 2026   Berndt's R-squared:                  0.6297
Time:                        18:07:41   Dhrymes's R-squared:                 0.5454
                                        Cov. Estimator:                      kernel
                                        Num. Constraints:                      None
                  Equation: JPM, Dependent Variable: ret_JPM                  
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          0.0002     0.0002     0.9213     0.3569     -0.0002      0.0006
d_DGS10     

In [78]:
import numpy as np
from scipy import stats

params = sur_results.params
cov = sur_results.cov

idx_jpm = params.index.get_loc("JPM_d_DGS10")
idx_bac = params.index.get_loc("BAC_d_DGS10")
idx_gs = params.index.get_loc("GS_d_DGS10")

R = np.zeros((2, len(params)))
R[0, idx_jpm] = 1
R[0, idx_bac] = -1
R[1, idx_jpm] = 1
R[1, idx_gs] = -1

r_vec = R @ params.values
cov_matrix = cov.values
wald_stat = r_vec @ np.linalg.inv(R @ cov_matrix @ R.T) @ r_vec

p_value = 1 - stats.chi2.cdf(wald_stat, df=2)

print(f"Wald statistic: {wald_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nJoint null hypothesis: JPM_beta = BAC_beta = GS_beta")
print(f"Reject at 5%: {p_value < 0.05}")

Wald statistic: 44.5255
P-value: 0.0000

Joint null hypothesis: JPM_beta = BAC_beta = GS_beta
Reject at 5%: True


In [79]:
def chow_test(data, break_date, y_col, x_cols):
    before = data[data.index < break_date]
    after = data[data.index >= break_date]
    
    def rss(subset):
        y = subset[y_col]
        X = sm.add_constant(subset[x_cols])
        model = sm.OLS(y, X).fit()
        return model.ssr, len(subset), model.df_model + 1
    
    rss_pooled, n_pooled, k = rss(data)
    rss_before, n1, _ = rss(before)
    rss_after, n2, _ = rss(after)
    
    numerator = (rss_pooled - (rss_before + rss_after)) / k
    denominator = (rss_before + rss_after) / (n1 + n2 - 2*k)
    f_stat = numerator / denominator
    p_value = 1 - stats.f.cdf(f_stat, k, n1 + n2 - 2*k)
    
    return f_stat, p_value

f_stat, p_val = chow_test(data, "2022-03-01", "ret_JPM", ["d_DGS10", "d_VIX", "ret_SP500"])
print(f"Chow test F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val:.4f}")

Chow test F-statistic: 24.0294
P-value: 0.0000


In [80]:
for bank, col in [("JPM", "ret_JPM"), ("BAC", "ret_BAC"), ("GS", "ret_GS")]:
    f_stat, p_val = chow_test(data, "2022-03-01", col, ["d_DGS10", "d_VIX", "ret_SP500"])
    print(f"{bank}: F-statistic = {f_stat:.4f}, p-value = {p_val:.4f}")

JPM: F-statistic = 24.0294, p-value = 0.0000
BAC: F-statistic = 56.4526, p-value = 0.0000
GS: F-statistic = 22.5153, p-value = 0.0000


In [81]:
desc_stats = data[["ret_JPM", "ret_BAC", "ret_GS", "ret_SP500", "d_DGS10", "d_DGS2", "d_VIX"]].describe().T
desc_stats = desc_stats[["mean", "std", "min", "max"]]
desc_stats.columns = ["Mean", "Std Dev", "Min", "Max"]
desc_stats

,Mean,Std Dev,Min,Max
ret_JPM,0.000708,0.016991,-0.162106,0.165621
ret_BAC,0.000515,0.019177,-0.167205,0.163786
ret_GS,0.000655,0.018575,-0.135881,0.161951
ret_SP500,0.000455,0.011196,-0.127652,0.090895
d_DGS10,0.000909,0.052970,-0.300000,0.290000
d_DGS2,0.001263,0.051833,-0.570000,0.340000
d_VIX,-0.001191,1.938726,-18.710003,24.860001


In [82]:
equations_full = {
    "JPM": {"dependent": data["ret_JPM"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "BAC": {"dependent": data["ret_BAC"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "GS": {"dependent": data["ret_GS"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX", "ret_SP500"]])},
    "SP500": {"dependent": data["ret_SP500"], "exog": sm.add_constant(data[["d_DGS10", "d_VIX"]])},
}

sur_model_full = SUR(equations_full)
sur_results_full = sur_model_full.fit(cov_type="kernel", kernel="bartlett", bandwidth=5)
params_full = sur_results_full.params
cov_full = sur_results_full.cov

def wald_single(param1_name, param2_name, params, cov):
    idx1 = params.index.get_loc(param1_name)
    idx2 = params.index.get_loc(param2_name)
    R = np.zeros((1, len(params)))
    R[0, idx1] = 1
    R[0, idx2] = -1
    r_vec = R @ params.values
    wald_stat = (r_vec @ np.linalg.inv(R @ cov.values @ R.T) @ r_vec)
    p_value = 1 - stats.chi2.cdf(wald_stat, df=1)
    return wald_stat, p_value

for bank in ["JPM", "BAC", "GS"]:
    w, p = wald_single(f"{bank}_d_DGS10", "SP500_d_DGS10", params_full, cov_full)
    print(f"{bank} vs SP500: Wald = {w:.4f}, p-value = {p:.4f}")

JPM vs SP500: Wald = 54.9275, p-value = 0.0000
BAC vs SP500: Wald = 96.3186, p-value = 0.0000
GS vs SP500: Wald = 11.5115, p-value = 0.0007


In [83]:
def run_regression(bank_return_col, data):
    y = data[bank_return_col]
    X = data[["d_DGS10", "d_VIX", "ret_SP500"]]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model

def run_regression_market(data):
    y = data["ret_SP500"]
    X = data[["d_DGS10", "d_VIX"]]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model

model_jpm = run_regression("ret_JPM", data)
model_bac = run_regression("ret_BAC", data)
model_gs = run_regression("ret_GS", data)
model_sp500_correct = run_regression_market(data)

In [84]:
def full_results_table(models, model_names):
    rows = []
    for name, model in zip(model_names, models):
        row = {
            "Variable": name,
            "const": f"{model.params['const']:.4f}",
            "d_DGS10": f"{model.params['d_DGS10']:.4f}***" if model.pvalues['d_DGS10'] < 0.01 else f"{model.params['d_DGS10']:.4f}",
            "d_VIX": f"{model.params['d_VIX']:.4f}",
            "ret_SP500": f"{model.params.get('ret_SP500', np.nan):.4f}" if 'ret_SP500' in model.params else "-",
            "R2": f"{model.rsquared:.3f}",
            "N": int(model.nobs),
        }
        rows.append(row)
    return pd.DataFrame(rows).set_index("Variable")

full_table = full_results_table(
    [model_jpm, model_bac, model_gs, model_sp500_correct],
    ["JPM", "BAC", "GS", "S&P 500"]
)
full_table

,const,d_DGS10,d_VIX,ret_SP500,R2,N
Variable,,,,,,
JPM,0.0002,0.0659***,-0.0001,1.0207,0.553,2914
BAC,-0.0001,0.0805***,0.0001,1.1601,0.535,2914
GS,0.0001,0.0367***,-0.0003,1.1476,0.551,2914
S&P 500,0.0004,0.0054,-0.0046,-,0.636,2914


In [85]:
for name, model in zip(["JPM", "BAC", "GS", "SP500"], [model_jpm, model_bac, model_gs, model_sp500_correct]):
    print(f"\n{name}:")
    print(model.bse)



JPM:
const        0.000199
d_DGS10      0.006533
d_VIX        0.000243
ret_SP500    0.046671
dtype: float64

BAC:
const        0.000242
d_DGS10      0.007243
d_VIX        0.000325
ret_SP500    0.061249
dtype: float64

GS:
const        0.000219
d_DGS10      0.007094
d_VIX        0.000248
ret_SP500    0.047478
dtype: float64

SP500:
const      0.000116
d_DGS10    0.005012
d_VIX      0.000198
dtype: float64


In [86]:
def run_regression_2y(bank_return_col, data):
    y = data[bank_return_col]
    X = data[["d_DGS2", "d_VIX", "ret_SP500"]]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model

def run_regression_market_2y(data):
    y = data["ret_SP500"]
    X = data[["d_DGS2", "d_VIX"]]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    return model

model_jpm_2y = run_regression_2y("ret_JPM", data)
model_bac_2y = run_regression_2y("ret_BAC", data)
model_gs_2y = run_regression_2y("ret_GS", data)
model_sp500_2y = run_regression_market_2y(data)

In [87]:
# 2Y robustness — standard errors
for name, model in zip(["JPM", "BAC", "GS", "SP500"], [model_jpm_2y, model_bac_2y, model_gs_2y, model_sp500_2y]):
    print(f"\n{name}:")
    print(model.bse)
    print(f"R2: {model.rsquared:.3f}, N: {int(model.nobs)}")


JPM:
const        0.000207
d_DGS2       0.005211
d_VIX        0.000256
ret_SP500    0.049469
dtype: float64
R2: 0.533, N: 2914

BAC:
const        0.000253
d_DGS2       0.006212
d_VIX        0.000356
ret_SP500    0.066672
dtype: float64
R2: 0.510, N: 2914

GS:
const        0.000222
d_DGS2       0.005926
d_VIX        0.000254
ret_SP500    0.048591
dtype: float64
R2: 0.546, N: 2914

SP500:
const     0.000116
d_DGS2    0.004198
d_VIX     0.000198
dtype: float64
R2: 0.635, N: 2914


In [88]:
residuals = pd.DataFrame({
    "JPM": model_jpm.resid,
    "BAC": model_bac.resid,
    "GS": model_gs.resid,
})
print(residuals.corr())

          JPM       BAC        GS
JPM  1.000000  0.742010  0.592091
BAC  0.742010  1.000000  0.579162
GS   0.592091  0.579162  1.000000
